# Episode 4: Probability, Entropy, and Perplexity
<a href="https://colab.research.google.com/github/Kinvecta/how-llms-work/blob/main/notebooks/Episode%2004%20-%20Probability%20Entropy%20Perplexity.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

**Companion notebook** for the series *How Large Language Models Work: From Math to Transformers*.

This notebook reproduces the Math Lab numbers from Episode 4, one step per section. It uses only `numpy` (already installed in Colab), so choose **Runtime, Run all** and nothing else is needed.

Covers: softmax, the shift check, entropy in nats and bits, perplexity, the lumped bucket, and the 0.97 example. The probabilities are the illustrative values from Episode 1 (after “The cat sat on the”), not measured from any model. Logs are natural unless a bits column says otherwise.

Every value that appears in the episode is checked with an `assert`. If a cell fails, a number in the script and the math have drifted apart.

## Setup

In [1]:
import numpy as np

words = ["mat", "floor", "sofa", "other"]
p = np.array([0.30, 0.20, 0.15, 0.35])      # illustrative, from Episode 1

def softmax(h):
    h = np.asarray(h, dtype=float)
    e = np.exp(h - h.max())                 # subtracting the largest score is the stable form
    return e / e.sum()

def entropy(p):
    p = np.asarray(p, dtype=float)
    return float(-np.sum(p * np.log(p)))    # nats

surprisal = lambda q: -np.log(q)            # -ln p, in nats

## Step 1: check the distribution

In [2]:
print("sum of p =", p.sum())
assert np.isclose(p.sum(), 1.0)

sum of p = 1.0


## Step 2: softmax returns it
Set h = ln p. Then softmax(h) = p exactly. A second example: h = (0, 1, 2).

In [3]:
h = np.log(p)
print("h =", np.round(h, 2))
print("softmax(h) =", np.round(softmax(h), 2))
assert np.allclose(softmax(h), p)

s2 = softmax([0, 1, 2])
print("softmax((0, 1, 2)) =", np.round(s2, 3))
assert np.allclose(np.round(s2, 3), [0.090, 0.245, 0.665])

h = [-1.2  -1.61 -1.9  -1.05]
softmax(h) = [0.3  0.2  0.15 0.35]
softmax((0, 1, 2)) = [0.09  0.245 0.665]


## Step 3: the shift check
Adding the same number s to every score changes nothing.

In [4]:
shifted = softmax(np.array([0, 1, 2]) + 5)
print("softmax((5, 6, 7)) =", np.round(shifted, 3))
assert np.allclose(shifted, s2)

softmax((5, 6, 7)) = [0.09  0.245 0.665]


## Step 4: entropy
Surprisals in nats and bits, then H(p) = sum of p·(−ln p).

In [5]:
for w, q in zip(words, p):
    print(f"{w:6s} p={q:.2f}  surprisal = {surprisal(q):.2f} nats = {surprisal(q)/np.log(2):.2f} bits")

H = entropy(p)
print(f"H(p) = {H:.3f} nats = {H/np.log(2):.2f} bits")
assert round(H, 3) == 1.335
assert round(H / np.log(2), 2) == 1.93

mat    p=0.30  surprisal = 1.20 nats = 1.74 bits
floor  p=0.20  surprisal = 1.61 nats = 2.32 bits
sofa   p=0.15  surprisal = 1.90 nats = 2.74 bits
other  p=0.35  surprisal = 1.05 nats = 1.51 bits
H(p) = 1.335 nats = 1.93 bits


## Step 5: perplexity of this distribution
exp(H) is close to 4 but not 4, because the outcomes are unequal. For c equally likely outcomes, exp(H) = c exactly.

In [6]:
print("exp(H) =", round(np.exp(H), 2))
assert round(np.exp(H), 2) == 3.80

c = 4
uniform = np.full(c, 1 / c)
print(f"Uniform over {c}: H = {entropy(uniform):.3f} nats = {entropy(uniform)/np.log(2):.0f} bits, exp(H) = {np.exp(entropy(uniform)):.0f}")
assert np.isclose(entropy(uniform), np.log(4)) and np.isclose(np.exp(entropy(uniform)), 4)

exp(H) = 3.8
Uniform over 4: H = 1.386 nats = 2 bits, exp(H) = 4


## Step 6: the lumped bucket
“Other” stands for many tokens. Splitting it adds 0.35 × (the entropy within “other”), which is zero or more, so 1.335 nats is a lower bound for the full vocabulary. Here we split “other” into k equally likely tokens as an illustration.

In [7]:
for k in (1, 2, 5, 20):
    within = np.log(k)                       # entropy of k equally likely tokens inside "other"
    full = H + 0.35 * within
    print(f"other split into {k:2d} tokens: H_full = {full:.3f} nats  (never below {H:.3f})")
    assert full >= H - 1e-12

# Direct check for k = 5: build the split distribution and compute its entropy
split = np.array([0.30, 0.20, 0.15] + [0.35 / 5] * 5)
assert np.isclose(entropy(split), H + 0.35 * np.log(5))

other split into  1 tokens: H_full = 1.335 nats  (never below 1.335)
other split into  2 tokens: H_full = 1.578 nats  (never below 1.335)
other split into  5 tokens: H_full = 1.898 nats  (never below 1.335)
other split into 20 tokens: H_full = 2.384 nats  (never below 1.335)


## Step 7: perplexity of one observed token
If the true next token is “mat,” the cross-entropy is −ln 0.30, and PP = exp(−ln 0.30) = 1/0.30.

In [8]:
ce = surprisal(0.30)
print(f"cross-entropy = {ce:.3f} nats, PP = {np.exp(ce):.2f}, 1/p = {1/0.30:.2f}")
assert round(ce, 3) == 1.204
assert round(np.exp(ce), 2) == 3.33

cross-entropy = 1.204 nats, PP = 3.33, 1/p = 3.33


## Misconception check: confidence is not correctness
A model gives 0.97 to “sofa” and 0.01 to each of the other three words. Its entropy is low. If the observed token is “mat,” its perplexity is 1/0.01 = 100.

In [9]:
q = np.array([0.01, 0.01, 0.97, 0.01])       # order: mat, floor, sofa, other
print(f"entropy = {entropy(q):.2f} nats (low), against {np.log(4):.2f} for the uniform model")
print("observed token = mat -> PP =", round(1 / q[0]))
assert round(entropy(q), 2) == 0.17
assert round(1 / q[0]) == 100

entropy = 0.17 nats (low), against 1.39 for the uniform model
observed token = mat -> PP = 100


## Check question
A model spreads probability evenly over four candidate words. (a) Entropy in nats and bits? (b) Perplexity? (c) A second model gives “mat” 0.97 and 0.01 to each of the other three. Higher or lower entropy?

In [10]:
print(f"(a) {np.log(4):.2f} nats = 2 bits   (b) {np.exp(np.log(4)):.0f}   (c) lower: {entropy(q):.2f} against {np.log(4):.2f}")
q2 = np.array([0.97, 0.01, 0.01, 0.01])
assert entropy(q2) < np.log(4)

(a) 1.39 nats = 2 bits   (b) 4   (c) lower: 0.17 against 1.39


In [11]:
print("All Episode 4 checks passed.")

All Episode 4 checks passed.
